# 04 — Modelling and evaluation (RQ1, RQ2)

**Input:** `features.csv` (from `03`). **Outputs:** `results_summary.csv`, `results_grouped_importance.csv`, `fig_04_summary.png`.

This notebook turns the leakage-safe feature matrix into defensible results. The evaluation is deliberately conservative:

- **Baselines first.** A prior-only floor and a logistic-regression baseline, so the gradient-boosting result means "beats these by X" rather than a lone number.
- **Tuning never touches test.** Hyperparameters are chosen on a *temporal* validation slice carved from the training years. The test set is scored once.
- **Embargo for every horizon.** For each horizon H, the training rows are only tracks whose H-day label window closes before the test period (`train_rows(H)`).
- **All three horizons** (6 / 12 / 24 months) are reported side by side.
- **Uncertainty** is shown with bootstrap 95 % confidence intervals on the test set.
- **Calibration** and **permutation importance** are reported, both per feature and per group of correlated features.
- **Robustness:** the model is re-evaluated without power users, and without tracks whose label is disputed.

In [1]:
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             precision_recall_curve)
from sklearn.inspection import permutation_importance
rng=np.random.default_rng(0)

d=pd.read_csv("data/processed/features.csv")
feat=[c for c in d.columns if c.startswith("f_")]
HZ=[180,365,730]; PRIMARY=365
nc=pd.read_csv("data/processed/nodes_clean.csv",usecols=["upload_id","date_unix"])
d=d.merge(nc,on="upload_id",how="left"); assert d.date_unix.notna().all()
tr_all=d[d.split=="train"]; te_all=d[d.split=="test"]
CUTOFF_TS=int(pd.Timestamp("2022-01-01",tz="UTC").timestamp()); DAY=86400
def train_rows(H):
    """training rows whose H-day label window closes before the test period"""
    gap=max(H,PRIMARY)*DAY
    return d[(d.date_unix<CUTOFF_TS-gap)&d[f"valid_{H}"]]
assert len(train_rows(365))==len(tr_all[tr_all.valid_365])
def xy(frame,H):
    f=frame[frame[f"valid_{H}"]]; return f[feat].values, f[f"y_{H}"].astype(int).values
def boot_ci(y, p, fns, n=1000):
    """95% percentile CIs for one or more metrics, on shared resamples."""
    idx = np.arange(len(y)); out = []
    for _ in range(n):
        s = rng.choice(idx, len(idx), replace=True)
        if 0 < y[s].sum() < len(s):
            out.append([f(y[s], p[s]) for f in fns])
    return np.percentile(np.array(out), [2.5, 97.5], axis=0).T
print("train",len(tr_all),"test",len(te_all),"| features",len(feat))

train 46065 test 3330 | features 51


## 1. Hyperparameter tuning on a temporal validation slice

The training years are split again in time:
- fitting on tracks posted **up to 2014** (n = 35,292),
- validating on tracks posted **2016–2020** (n = 7,905),
- leaving **2015 out entirely** as an embargo, so no validation label window overlaps the fitting data.

The best gradient-boosting setting is chosen on this validation slice only. Across a grid of 6 settings, the best was depth 3 with learning rate 0.05 (validation AUC 0.821).

*Note: the printed labels "(<= 2015)" / "(> 2015)" in the output below are outdated. The actual split is ≤ 2014 / ≥ 2016.*

In [2]:
cut=int(tr_all.f_post_year.quantile(0.80))
fit=tr_all[tr_all.f_post_year<cut]; val=tr_all[tr_all.f_post_year>cut]   # year `cut` is the embargo
Xf,yf=fit[feat].values, fit[f"y_{PRIMARY}"].astype(int).values
Xv,yv=val[feat].values, val[f"y_{PRIMARY}"].astype(int).values
grid=[dict(max_depth=md,learning_rate=lr,max_iter=300,l2_regularization=1.0)
      for md in [3,4,6] for lr in [0.05,0.1]]
best=None
for g in grid:
    m=HistGradientBoostingClassifier(random_state=0,**g).fit(Xf,yf)
    a=roc_auc_score(yv,m.predict_proba(Xv)[:,1])
    best=(a,g) if best is None or a>best[0] else best
BEST=best[1]
print(f"validation fit n={len(fit)} (<= {int(cut)}), val n={len(val)} (> {int(cut)})")
print(f"best validation AUC={best[0]:.3f}  ->  {BEST}")

validation fit n=35292 (<= 2015), val n=7905 (> 2015)
best validation AUC=0.821  ->  {'max_depth': 3, 'learning_rate': 0.05, 'max_iter': 300, 'l2_regularization': 1.0}


## 2. Main results — baselines vs tuned model, across all three horizons

`prior_only` predicts the training base rate for everyone (AUC 0.5 by definition, AP =
base rate). `logreg` is a standardised linear baseline. `hgb_tuned` is the gradient
boosting model with the settings chosen above. AUC and AP intervals are 95 % bootstrap, computed on shared resamples.

In [3]:
rows=[]; PRED={}
for H in HZ:
    Xtr,ytr=xy(train_rows(H),H); Xte,yte=xy(te_all,H); base=yte.mean()
    sc=StandardScaler().fit(Xtr)
    plr=LogisticRegression(max_iter=3000).fit(sc.transform(Xtr),ytr).predict_proba(sc.transform(Xte))[:,1]
    hgb=HistGradientBoostingClassifier(random_state=0,**BEST).fit(Xtr,ytr)
    ph=hgb.predict_proba(Xte)[:,1]; PRED[H]=(yte,ph,plr)
    for name,p,has in [("prior_only",np.full(len(yte),ytr.mean()),False),
                       ("logreg",plr,True),("hgb_tuned",ph,True)]:
        au=roc_auc_score(yte,p) if has else 0.5
        ap=average_precision_score(yte,p)
        if has:
            (lo,hi),(alo,ahi)=boot_ci(yte,p,[roc_auc_score,average_precision_score])
        else:
            lo=hi=alo=ahi=np.nan
        rows.append(dict(horizon_days=H,model=name,test_n=len(yte),base_rate=round(base,3),
              AUC=round(au,3),AUC_lo=round(lo,3),AUC_hi=round(hi,3),
              AP=round(ap,3),AP_lo=round(alo,3),AP_hi=round(ahi,3)))
res=pd.DataFrame(rows); res.to_csv("results_summary.csv",index=False)
print(res.to_string(index=False))

 horizon_days      model  test_n  base_rate   AUC  AUC_lo  AUC_hi    AP  AP_lo  AP_hi
          180 prior_only    3330      0.296 0.500     NaN     NaN 0.296    NaN    NaN
          180     logreg    3330      0.296 0.805   0.790   0.821 0.600  0.567  0.635
          180  hgb_tuned    3330      0.296 0.830   0.815   0.844 0.630  0.596  0.664
          365 prior_only    3330      0.331 0.500     NaN     NaN 0.331    NaN    NaN
          365     logreg    3330      0.331 0.812   0.797   0.828 0.638  0.607  0.671
          365  hgb_tuned    3330      0.331 0.842   0.828   0.855 0.685  0.655  0.714
          730 prior_only    2344      0.375 0.500     NaN     NaN 0.375    NaN    NaN
          730     logreg    2344      0.375 0.824   0.807   0.841 0.711  0.678  0.747
          730  hgb_tuned    2344      0.375 0.861   0.846   0.876 0.758  0.729  0.789


**Reading the results.** Discrimination is strong at every horizon:

| Horizon | Tuned gradient boosting | Logistic regression |
|---|---|---|
| 6 months | AUC 0.830 [0.815, 0.844] | AUC 0.805 |
| 12 months | AUC 0.842 [0.828, 0.855], AP 0.685 against a base rate of 0.331 | AUC 0.812 |
| 24 months | AUC 0.861 [0.846, 0.876], AP 0.758 | AUC 0.824 |

Discrimination *rises* with the horizon rather than falling. The later remixes are still predictable from signals available at posting time. The horizons are not evaluated on identical data (the 24-month horizon has a shorter training set and only 2,344 test tracks), so the rise should be read with caution. Both learned models beat the floor comfortably, and gradient boosting beats the linear baseline by about 3–4 AUC points at every horizon.

## 3. Calibration (primary horizon)

Good ranking (AUC) does not guarantee that the predicted probabilities are *accurate*. The reliability table compares predicted and observed remix rates for each decile of predicted score. The Brier score summarises probability accuracy (lower is better).

**Result.** The Brier score is 0.154, and the model **systematically over-predicts** (predicted > observed in 9 of 10 deciles):
- in the lowest deciles it predicts about twice the observed rate (0.057 vs 0.024),
- in the top decile it predicts 0.874 against 0.736 observed.

The model ranks tracks well, but its raw probabilities are inflated. Post-hoc calibration (isotonic or Platt) would be needed before the probabilities could be used for decisions.

In [4]:
yte,ph,_=PRED[PRIMARY]
brier=brier_score_loss(yte,ph)
bins=pd.qcut(ph,10,duplicates="drop")
cal=pd.DataFrame({"p":ph,"y":yte}).groupby(bins,observed=True).agg(
        predicted=("p","mean"),observed=("y","mean"),n=("y","size"))
print(f"Brier={brier:.3f}"); print(cal.round(3).to_string())
print("\nNote: the model is over-confident in its top deciles (predicted > observed) —",
      "it ranks well but raw probabilities are inflated. Isotonic/Platt calibration on",
      "a held-out slice would fix this if calibrated probabilities are needed.")

Brier=0.154
                  predicted  observed    n
(0.0322, 0.0652]      0.057     0.024  333
(0.0652, 0.0892]      0.074     0.036  333
(0.0892, 0.123]       0.106     0.048  333
(0.123, 0.21]         0.149     0.102  333
(0.21, 0.344]         0.275     0.195  333
(0.344, 0.445]        0.398     0.399  333
(0.445, 0.569]        0.498     0.483  333
(0.569, 0.719]        0.657     0.613  333
(0.719, 0.822]        0.777     0.673  333
(0.822, 0.969]        0.874     0.736  333

Note: the model is over-confident in its top deciles (predicted > observed) — it ranks well but raw probabilities are inflated. Isotonic/Platt calibration on a held-out slice would fix this if calibrated probabilities are needed.


## 4. What drives the prediction — permutation importance (RQ1)

Permutation importance measures how much test AUC drops when one feature is shuffled. It is computed on the test set, so it reflects what matters on unseen data.

**Caveat.** When features are correlated, importance is *diluted* across them. For example, `n_sources`, `in_degree_local` and `is_derivative` all describe how many sources a track uses, so each looks less important alone than the property really is. This is why `is_derivative`, the strongest single feature in `03`, does not appear in the top 12 here. The grouped version below is the more reliable summary.

**Result (single features).** `n_sources` 0.118, `num_files` 0.090, `author_prior_success_rate` 0.020, `author_prior_uploads` 0.009. Everything else is below 0.005.

In [5]:
hgb=HistGradientBoostingClassifier(random_state=0,**BEST).fit(*xy(train_rows(PRIMARY),PRIMARY))
Xte,yte=xy(te_all,PRIMARY)
pi=permutation_importance(hgb,Xte,yte,scoring="roc_auc",n_repeats=10,random_state=0)
imp=pd.Series(pi.importances_mean,index=feat).sort_values(ascending=False)
print("top-12 (mean drop in test AUC when shuffled):"); print(imp.head(12).round(4).to_string())
print("\nThe track's own structure — number of sources it draws on and number of files",
      "it ships (plausibly stems/multitracks that make it easy to remix) — dominates,",
      "ahead of author track-record. That is a substantive RQ1 finding worth probing.")

top-12 (mean drop in test AUC when shuffled):
f_n_sources                           0.1183
f_num_files                           0.0901
f_author_prior_success_rate           0.0198
f_author_prior_uploads                0.0092
f_author_prior_remixes_received       0.0040
f_sys_flac                            0.0026
f_author_prior_remixes_given          0.0024
f_tag_female_vocals                   0.0010
f_author_prior_remixed_track_count    0.0009
f_tag_instrumental                    0.0009
f_lic_other                           0.0007
f_parent_mean_prior_remixes           0.0006

The track's own structure — number of sources it draws on and number of files it ships (plausibly stems/multitracks that make it easy to remix) — dominates, ahead of author track-record. That is a substantive RQ1 finding worth probing.


### 4b. Grouped permutation importance (RQ1 headline)

Correlated features are shuffled together, so a property's importance is not split across its duplicates. Each group is shuffled 20 times; the table shows the mean drop in AUC with a 95 % range.

**Result.**

| Feature group | Drop in AUC |
|---|---|
| Source structure | **0.113** [0.104, 0.123] |
| Files / format | **0.082** [0.076, 0.090] |
| Author history | **0.029** [0.023, 0.033] |
| Content tags | 0.003 |
| Source prominence | 0.001 |
| Licence | 0.001 |
| Title | ≈ 0 |
| Posting time | ≈ 0 |

**RQ1 answer.** How a track is built matters about **4× more** (source structure) and **3× more** (files) than who built it. Author track record still contributes clearly. Tags, licence, title and posting time add almost nothing.

In [6]:
# Grouped permutation importance: shuffle correlated features together
groups={
 "source structure (n_sources, in_degree, is_derivative)":["f_n_sources","f_in_degree_local","f_is_derivative"],
 "source prominence":["f_parent_max_prior_remixes","f_parent_mean_prior_remixes"],
 "files / format (num_files, flac)":["f_num_files","f_sys_flac"],
 "author history":[c for c in feat if c.startswith("f_author_")],
 "content tags":[c for c in feat if c.startswith("f_tag_")]+["f_n_usertags","f_has_usertags"],
 "licence":[c for c in feat if c.startswith("f_lic_")],
 "title":["f_title_len","f_title_words"],
 "posting time":["f_post_year","f_post_month","f_post_dow"],
}
assert sorted(sum(groups.values(),[]))==sorted(feat), "every feature must be in exactly one group"
te=te_all[te_all[f"valid_{PRIMARY}"]]
X=te[feat].copy(); y=te[f"y_{PRIMARY}"].astype(int).values
base=roc_auc_score(y,hgb.predict_proba(X.values)[:,1])
r=np.random.default_rng(0); out={}
for g,cols in groups.items():
    drops=[]
    for _ in range(20):
        Xp=X.copy(); Xp[cols]=Xp[cols].values[r.permutation(len(Xp))]
        drops.append(base-roc_auc_score(y,hgb.predict_proba(Xp.values)[:,1]))
    out[g]=dict(dAUC=np.mean(drops),lo=np.percentile(drops,2.5),hi=np.percentile(drops,97.5))
gimp=pd.DataFrame(out).T.sort_values("dAUC",ascending=False)
print(gimp.round(4).to_string()); gimp.to_csv("results_grouped_importance.csv")

                                                          dAUC      lo      hi
source structure (n_sources, in_degree, is_derivative)  0.1131  0.1035  0.1229
files / format (num_files, flac)                        0.0818  0.0756  0.0904
author history                                          0.0285  0.0229  0.0327
content tags                                            0.0030  0.0018  0.0044
source prominence                                       0.0010  0.0001  0.0019
licence                                                 0.0008  0.0001  0.0011
title                                                   0.0004 -0.0001  0.0009
posting time                                           -0.0006 -0.0014  0.0001


## 5. Power-user robustness

A few very prolific accounts dominate the data: **49 authors (the top 1 %, each with 198 or more tracks) produced 40.9 % of all tracks and 63 % of the test set (2,089 of 3,330).** If the headline result depended entirely on them, it would not generalise.

**Result.**
- Excluding power users, AUC is **0.807** (n = 1,241, base rate 0.387).
- On power users alone, AUC is 0.859 (n = 2,089).

The signal is not an artefact of a few accounts, although those accounts are somewhat more predictable. No confidence interval is computed for these subsets.

In [7]:
ac=d.groupby("user_name").size(); thr=ac.quantile(0.99)
power=set(ac[ac>=thr].index)
print(f"power users = authors with >= {int(thr)} tracks (top 1%): {len(power)} authors, "
      f"{100*d.user_name.isin(power).mean():.1f}% of all rows")
te=te_all[te_all[f"valid_{PRIMARY}"]].copy()
te["p"]=hgb.predict_proba(te[feat].values)[:,1]; te["y"]=te[f"y_{PRIMARY}"].astype(int)
for lab,m in [("all test",np.ones(len(te),bool)),
              ("excluding power users",~te.user_name.isin(power).values),
              ("power users only", te.user_name.isin(power).values)]:
    s=te[m]
    if s.y.nunique()<2: print(f"  {lab}: n={len(s)} single-class"); continue
    print(f"  {lab:22s} n={len(s):5d}  base={s.y.mean():.3f}  "
          f"AUC={roc_auc_score(s.y,s.p):.3f}  AP={average_precision_score(s.y,s.p):.3f}")
print("\nExcluding the top 1% of authors, AUC stays ~0.81 — the signal is not an artefact",
      "of a few prolific accounts, though those accounts are slightly more predictable.")

power users = authors with >= 196 tracks (top 1%): 49 authors, 40.9% of all rows
  all test               n= 3330  base=0.331  AUC=0.842  AP=0.685
  excluding power users  n= 1241  base=0.387  AUC=0.807  AP=0.682
  power users only       n= 2089  base=0.298  AUC=0.859  AP=0.691

Excluding the top 1% of authors, AUC stays ~0.81 — the signal is not an artefact of a few prolific accounts, though those accounts are slightly more predictable.


## 6. Summary figure

The four panels show:
- precision–recall curves by horizon,
- tuned-model AUC with 95 % confidence intervals,
- the calibration curve,
- the top-8 single-feature permutation importances.

In [8]:
fig,ax=plt.subplots(2,2,figsize=(11,8))
# (a) PR curves per horizon
for H in HZ:
    yt,ph,_=PRED[H]; pr,rc,_=precision_recall_curve(yt,ph)
    ax[0,0].plot(rc,pr,label=f"{H//30}mo (AP={average_precision_score(yt,ph):.2f})")
ax[0,0].set(title="Precision–Recall by horizon",xlabel="Recall",ylabel="Precision"); ax[0,0].legend()
# (b) AUC with CIs
h=res[res.model=="hgb_tuned"]
ax[0,1].errorbar([f"{x//30}mo" for x in h.horizon_days],h.AUC,
    yerr=[h.AUC-h.AUC_lo,h.AUC_hi-h.AUC],fmt="o-",capsize=4)
ax[0,1].set(title="Tuned model AUC (95% CI)",ylim=(0.75,0.9),ylabel="Test AUC")
# (c) calibration
ax[1,0].plot([0,1],[0,1],"k--",lw=1); ax[1,0].plot(cal.predicted,cal.observed,"o-")
ax[1,0].set(title=f"Calibration (Brier={brier:.3f})",xlabel="Predicted",ylabel="Observed")
# (d) importance
top=imp.head(8)[::-1]
ax[1,1].barh([c.replace('f_','') for c in top.index],top.values)
ax[1,1].set(title="Permutation importance (ΔAUC)")
plt.tight_layout(); plt.savefig("fig_04_summary.png",dpi=130,bbox_inches="tight")
print("saved fig_04_summary.png")

saved fig_04_summary.png


## 7. Label-noise sensitivity

The site counter reports a remix for 1,387 tracks that have no child in the graph (`counter_only_positive`; see `02`). The counter has no dates, so it cannot serve as an alternative horizon label. Instead, we exclude these disputed tracks and re-evaluate.

**Result.** The disputed tracks make up 2.9 % of training and 1.0 % of test. Excluding them changes almost nothing:
- excluded from test only: AUC 0.842 → **0.843**, AP 0.685 → 0.689;
- excluded from training and test: AUC **0.843**, AP 0.688.

The choice of label source does not drive the results.

In [9]:
# Sensitivity: exclude tracks where the site counter reports a remix the graph can't confirm
ph=pd.read_csv("data/processed/nodes_clean.csv",usecols=["upload_id","counter_only_positive"])
dd=d.merge(ph,on="upload_id",how="left")
trP=dd[(dd.date_unix<CUTOFF_TS-PRIMARY*DAY)&dd[f"valid_{PRIMARY}"]]
teP=dd[(dd.split=="test")&dd[f"valid_{PRIMARY}"]]
print(f"phantom share: train {trP.counter_only_positive.mean():.3f}  test {teP.counter_only_positive.mean():.3f}")
rows=[]
for lab,tr_,te_ in [("baseline",trP,teP),
                    ("test without phantoms",trP,teP[~teP.counter_only_positive]),
                    ("train+test without phantoms",trP[~trP.counter_only_positive],teP[~teP.counter_only_positive])]:
    m=HistGradientBoostingClassifier(random_state=0,**BEST).fit(tr_[feat].values,tr_[f"y_{PRIMARY}"].astype(int))
    p=m.predict_proba(te_[feat].values)[:,1]; y=te_[f"y_{PRIMARY}"].astype(int).values
    rows.append(dict(setting=lab,train_n=len(tr_),test_n=len(te_),AUC=round(roc_auc_score(y,p),3),AP=round(average_precision_score(y,p),3)))
print(pd.DataFrame(rows).to_string(index=False))

phantom share: train 0.029  test 0.010
                    setting  train_n  test_n   AUC    AP
                   baseline    46065    3330 0.842 0.685
      test without phantoms    46065    3296 0.843 0.689
train+test without phantoms    44745    3296 0.843 0.688


## 8. Findings and limitations (for the write-up)

**Findings (RQ2).**
- Signals available at posting time predict one-year remixing with AUC 0.842 [0.828, 0.855] (0.830–0.861 across 6/12/24-month horizons).
- This is well above the prior-only and logistic-regression baselines, under a strict temporal split with a horizon-specific embargo.

**Findings (RQ1).**
- The strongest drivers are the track's own structure: whether and how many sources it builds on (grouped ΔAUC 0.113), and how many files it ships (0.082).
- Both come well ahead of author history (0.029).
- Remixability appears to be largely *built in at upload*, although author track record still matters.

**Robustness.** The result holds on test tracks by authors outside the most prolific 1 % (AUC 0.807; the model is still trained on all authors) and when tracks with disputed labels are excluded (AUC 0.843)

**Limitations.**
1. **Association, not causation.** The importances describe what the model relies on, not causal effects.
2. **Calibration.** The model ranks better than it calibrates; raw probabilities are systematically too high.
3. **Single test window.** The test set is one 2022–mid-2025 slice with real covariate shift from the training era.
4. **Author concentration.** 49 accounts produced 41 % of the data.
5. **Horizon bounds.** The outcome is limited to the horizon, so late remixes are out of scope by design.
6. **Small tuning grid.** Only 6 settings were tried, and a single model family (histogram gradient boosting) was used.
7. **Missing engagement signals.** Community and engagement signals (e.g. `num_scores`) were not used, because only current snapshot values exist.
8. **Crawl dependence.** Everything rests on the crawl being complete and its remix links being accurate.

**Next.** `05` / `05b` test whether network structure adds anything beyond these node features, and `06` / `07` study source recommendation (RQ3).